# DeepTrace – Isolation Forest

This notebook implements the first stage of the DeepTrace anomaly detection pipeline using Isolation Forest. The model is trained on engineered cybersecurity features to identify anomalous user behavior without requiring labeled attack data.

In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

In [8]:
X = pd.read_csv("/content/deeptrace_features.csv")

y = pd.read_csv("/content/deeptrace_binary_labels.csv").squeeze()

In [9]:
print("Features Shape :", X.shape)
print("Labels Shape   :", y.shape)

X.head()

Features Shape : (239480, 49)
Labels Shape   : (239480,)


,department,role,office_location,device,network_type,authentication,event_type,application,resource,risk_baseline,...,upload_count,database_query_count,file_access_count,admin_action_count,powershell_execution_count,is_privileged_user,is_admin,remote_access,high_risk_event,foreign_login
0,1.0,0.45,0.8,0.25,0.0,1.000000,0.500000,0.03125,0.045455,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.45,0.8,0.25,0.0,0.333333,0.000000,0.00000,0.409091,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,0.45,0.8,0.25,0.0,1.000000,0.083333,0.90625,0.636364,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.45,0.8,0.25,0.0,0.000000,0.083333,0.21875,0.090909,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.45,0.8,0.25,0.0,0.666667,0.083333,0.37500,0.636364,0.5,...,0.0,0.1875,0.0,0.230769,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
iso_forest = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X)

print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [43]:
predictions = iso_forest.predict(X)

predictions = np.where(
    predictions == -1,
    1,
    0
)

In [44]:
pd.Series(predictions).value_counts()

,count
0,227506
1,11974


In [45]:
accuracy = accuracy_score(y, predictions)
precision = precision_score(y, predictions)
recall = recall_score(y, predictions)
f1 = f1_score(y, predictions)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.9332
Precision: 0.0630
Recall   : 0.1365
F1 Score : 0.0862


In [46]:
cm = confusion_matrix(y, predictions)

print("Confusion Matrix\n")
print(cm)


Confusion Matrix

[[222737  11220]
 [  4769    754]]


In [47]:
print(classification_report(y, predictions))

              precision    recall  f1-score   support

           0       0.98      0.95      0.97    233957
           1       0.06      0.14      0.09      5523

    accuracy                           0.93    239480
   macro avg       0.52      0.54      0.53    239480
weighted avg       0.96      0.93      0.95    239480



In [48]:
anomaly_scores = -iso_forest.decision_function(X)

print("Minimum Score :", anomaly_scores.min())
print("Maximum Score :", anomaly_scores.max())
print("Mean Score    :", anomaly_scores.mean())

Minimum Score : -0.16107398054822614
Maximum Score : 0.09005052086888377
Mean Score    : -0.07161205447501665


In [49]:
# Generate predictions
predictions = iso_forest.predict(X)
predictions = np.where(predictions == -1, 1, 0)

# Generate anomaly scores
anomaly_scores = -iso_forest.decision_function(X)

In [50]:
import joblib

joblib.dump(
    iso_forest,
    "models/trained/isolation_forest.pkl"
)


['models/trained/isolation_forest.pkl']

In [51]:
isolation_output = pd.DataFrame({
    "anomaly_score": anomaly_scores,
    "isolation_prediction": predictions
})

isolation_output.to_csv(
    "data/processed/isolation_forest_output.csv",
    index=False
)

In [52]:
transformer_input = X.copy()

transformer_input["anomaly_score"] = anomaly_scores
transformer_input["if_prediction"] = predictions

transformer_input.to_csv(
    "data/processed/deeptrace_transformer_input.csv",
    index=False
)